In [ ]:
# LangGraph_nodes.py
# 定义 AgentState 与所有节点工厂函数,用于构建法律问答的 LangGraph 工作流

from typing import Dict, List, Any, Callable

from langchain_core.language_models import BaseLanguageModel
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel


class AgentState(BaseModel):
    # 用户问题
    query: str

    # 历史消息 分角色user agent : content
    messages: List[Dict[str, str]]

    # 是否为法律问题
    is_law_questions: bool

    # 是否为简单问题
    is_simple_questions: bool

    # RAG检索出的内容
    rag_retrieved: str

    # 评估等级 Correct Ambiguous Incorrect
    evaluation: Dict[str, List[Dict]]

    # 联网搜索结果
    web_search_results: List[str]

    # CRAG拼接检索结果
    crag_context: str

    # 上下文组装提示词送入llm
    final_prompts: str

    # 最终的回答
    final_answer: str

    # 是否需要pdf形式输出
    is_pdf_output: bool

    # 循环下一轮回答
    should_loop: bool


def start_node(state: AgentState) -> dict:
    """
    开始节点:获取用户问题,加入对话历史,重置本轮检索与评估字段
    预期 state 已包含新 query 和以往的 messages
    :param state:
    :return:
    """
    # 读取并通过副本追加新消息（避免原地修改）
    messages = state.messages.copy()
    if state.query.strip():
        messages.append({"role": "user", "content": state.query})

    return {
        "messages": messages,
        "is_law_questions": False,
        "is_simple_questions": False,
        "rag_retrieved": "",
        "evaluation": {"correct": [], "ambiguous": [], "incorrect": []},
        "web_search_results": [],
        "crag_context": "",
        "final_prompts": "",
        "final_answer": "",
        "should_loop": True,
    }


def create_router_node(llm: BaseLanguageModel) -> Callable:
    """
    创建路由节点
    分辨简单问题还是法律相关问题
    :param llm:
    :return:
    """

    ROUTER_PROMPT = PromptTemplate.from_template("""
    你是一个法律咨询系统的任务路由器。请分析用户问题,判断：

    1. **是否法律问题** (`is_legal`)：true 或 false
       - 法律问题：涉及法律概念、法条、案例、权利义务、纠纷解决等。
       - 非法律问题：日常闲聊、非法律领域知识。

    2. **是否简单问题** (`is_simple`)：true 或 false
       - 简单问题：无需检索法条或案例,凭常识或基础法律知识即可回答（包括非法律问题一律视为简单）。
       - 复杂问题：需要查阅具体法条、司法解释、相关判例或深度推理。

    严格输出 JSON 格式：
    {{"is_legal": true/false, "is_simple": true/false}}

    示例：
    用户：今天天气怎么样？ -> {{"is_legal": false, "is_simple": true}}
    用户：什么是合同？ -> {{"is_legal": true, "is_simple": true}}
    用户：怎么煮面条？ -> {{"is_legal": false, "is_simple": true}}
    用户：合同需要书面形式吗？ -> {{"is_legal": true, "is_simple": true}}
    用户：你好 -> {{"is_legal": false, "is_simple": true}}
    用户：婚姻中一方隐匿财产,离婚时如何分割？ -> {{"is_legal": true, "is_simple": false}}
    用户：我被公司无故辞退,可以要求多少赔偿？ -> {{"is_legal": true, "is_simple": false}}
    用户：商标侵权和不正当竞争的区别是什么？如何取证？ -> {{"is_legal": true, "is_simple": false}}

    现在请判断：
    {query}
    """)
    chain = ROUTER_PROMPT | llm | StrOutputParser()

    def router_node(state: AgentState) -> dict:
        query = state.query  # Pydantic 属性访问
        raw = chain.invoke({"query": query})
        try:
            result = json.loads(raw)
            is_legal = result.get("is_legal", False)
            is_simple = result.get("is_simple", False) if is_legal else False
        except Exception:
            is_legal = False
            is_simple = False

        # 映射到 AgentState 的字段
        return {
            "is_law_questions": is_legal,
            "is_simple_questions": is_simple,
        }

    return router_node


def simple_llm_node(state: AgentState, llm: BaseLanguageModel):
    """
    回答简单问题的节点
    :param state:
    :param llm:
    :return:
    """
    SIMPLE_PROMPT = PromptTemplate.from_template(
        """
        你是一个智能,可靠,友善的AI助手。请遵循以下原则
        1. 准确优先:不确定时不编造,明确说“我不知道”或“需要进一步信息”
        2. 简洁清晰:直接回答问题核心,避免啰嗦和无关内容
        3. 安全中立:不生成暴力,色情,仇恨或歧视性内容；不提供违法建议
        4. 结构友好:当需要分点,列表或步骤时,使用清晰的结构（如序号或短横）
        5. 仅输出回答内容:不要添加“作为AI模型...”之类的开场白或免责声明,除非用户明确询问你的身份
        """
    )
    chain = SIMPLE_PROMPT | llm | StrOutputParser()

    answer = chain.invoke({"query": state.query})
    state.final_answer = answer
    state.messages.append({"role": "assistant", "content": answer})


def create_retrieval_node(
        rag_service: Any,
        namespace: str = "law_cases",
        top_k: int = 20,
        rerank_top_n: int = 10,
        alpha: float = 0.5,
) -> Callable[[Dict[str, Any]], Dict[str, Any]]:
    """
    返回 RAG 检索节点,使用稠密+稀疏混合检索,结果存入 state['rag_retrieved']

    :param rag_service: RAG 服务实例,需实现 search_withDenseSparse 方法
    :param namespace: 向量库命名空间
    :param top_k: 初筛返回的文档数（给重排器）
    :param rerank_top_n: 重排后最终保留的文档数
    :param alpha: 稠密检索与稀疏检索的权重平衡因子 (0=纯稀疏, 1=纯稠密)
    :return: 节点函数,接受 state 字典,返回包含 rag_retrieved 的更新字典
    """

    def retrieval_node(state: Dict[str, Any]) -> Dict[str, Any]:
        query = state.get("query", "")
        if not query:
            return {"rag_retrieved": []}

        # 调用混合检索方法
        docs = rag_service.search_withDenseSparse(
            query=query,
            namespace=namespace,
            top_k=top_k,
            rerank_top_n=rerank_top_n,
            alpha=alpha,
        )

        # 将检索结果转换为可序列化的字典列表
        retrieved = []
        for doc in docs:
            # 假设 doc 具备 id, metadata, 以及可能存在的 rerank_score 属性
            retrieved.append({
                "id": getattr(doc, "id", None),
                "chunk_text": doc.metadata.get("chunk_text", "") if hasattr(doc, "metadata") else "",
                "rerank_score": getattr(doc, "rerank_score", None),
                "score": getattr(doc, "score", None),
                "metadata": getattr(doc, "metadata", {}),
            })

        return {"rag_retrieved": retrieved}

    return retrieval_node


def create_evaluate_node(llm: BaseLanguageModel, correct_threshold: float = 0.7,
                         incorrect_threshold: float = 0.3) -> Callable:
    """
    创建评估节点
    评估RAG检索节点的结果
    分为: Correct Ambiguous Incorrect
    :param llm: 
    :param correct_threshold: 
    :param incorrect_threshold: 
    :return: 
    """

    def evaluate_node(state: AgentState) -> dict:
        docs = state.get("retrieved_docs", [])
        correct, ambiguous, incorrect = [], [], []
        for doc in docs:
            score = doc.get("rerank_score", 0.0)
            if score >= correct_threshold:
                correct.append(doc)
            elif score < incorrect_threshold:
                incorrect.append(doc)
            else:
                ambiguous.append(doc)
        return {"evaluation": {"correct": correct, "ambiguous": ambiguous, "incorrect": incorrect}}

    return evaluate_node


def create_web_search_node(search_func: Callable[[str, int], List[str]]) -> Callable:
    """
    返回联网检索节点:对 ambiguous 和 incorrect 的内容进行外部搜索。
    search_func(query, num) -> List[str] 返回片段列表。
    """

    def web_search_node(state: AgentState) -> dict:
        evaluation = state.get("evaluation", {})
        # 混合需要补充知识的文档文本,构建搜索查询
        docs_to_search = evaluation.get("ambiguous", []) + evaluation.get("incorrect", [])
        if not docs_to_search:
            return {"web_results": []}
        # 简单拼接文档前200字符作为二次搜索关键词,或用原query
        search_query = state["query"]  # 直接用用户问题
        results = search_func(search_query, num=5)
        return {"web_results": results}

    return web_search_node


def create_analysis_node(llm: BaseLanguageModel) -> Callable:
    """
    LLM分析节点:组装 correct + ambiguous + web_results 为上下文,生成最终答案。
    """

    def analysis_node(state: AgentState) -> dict:
        evaluation = state.get("evaluation", {})
        correct = evaluation.get("correct", [])
        ambiguous = evaluation.get("ambiguous", [])
        web_results = state.get("web_results", [])

        # 构建上下文
        context_parts = []
        for doc in correct:
            context_parts.append(f"[高相关案例] {doc['chunk_text']}")
        for doc in ambiguous:
            context_parts.append(f"[中等相关案例] {doc['chunk_text']}")
        for i, snippet in enumerate(web_results, 1):
            context_parts.append(f"[外部资料{i}] {snippet}")
        context = "\n\n".join(context_parts)

        prompt = PromptTemplate.from_template(
            "你是资深法律顾问。请根据以下资料回答用户问题。\n"
            "资料:\n{context}\n\n"
            "用户问题:{query}\n"
            "详细分析并给出法律建议:"
        )
        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "query": state["query"]})

        messages = state.get("messages", [])
        messages.append({"role": "assistant", "content": answer})
        return {
            "crag_context": context,
            "answer": answer,
            "messages": messages
        }

    return analysis_node


def create_postprocess_node(pdf_generator: Callable[[str, str], str]) -> Callable:
    """
    后处理节点:将最终回答保存为PDF,并更新pdf_path。
    pdf_generator(answer_text, output_filename) -> pdf_path
    """

    def postprocess_node(state: AgentState) -> dict:
        answer = state.get("answer", "")
        pdf_path = pdf_generator(answer, f"report_{state['query'][:20]}.pdf")
        return {"pdf_path": pdf_path}

    return postprocess_node


# 可选:记忆更新 / 循环控制节点
def memory_update_node(state: AgentState) -> dict:
    """
    保留记忆并准备下一轮。将 should_continue 置为 True 可继续循环。
    """
    # 简单重置部分字段,保留messages历史
    return {
        "query": "",  # 清空查询,等待下一轮输入
        "should_continue": True,
        "is_legal": False,
        "difficulty": "",
        "retrieved_docs": [],
        "evaluation": {"correct": [], "ambiguous": [], "incorrect": []},
        "web_results": [],
        "crag_context": "",
        "answer": "",
        "pdf_path": ""
    }


# 条件路由函数（必须单独定义,因为条件边需要引用函数名称,此处提供样例）
def route_by_difficulty(state: AgentState) -> str:
    if not state["is_legal"]:
        return "non_legal"
    if state["difficulty"] == "简单":
        return "simple_llm"
    else:
        return "crag_retrieval"  # 困难或其它情况进入CRAG检索


def route_after_evaluation(state: AgentState) -> str:
    eval = state.get("evaluation", {})
    correct_count = len(eval.get("correct", []))
    ambiguous_count = len(eval.get("ambiguous", []))
    incorrect_count = len(eval.get("incorrect", []))
    # 如果正确文档足够,直接分析；否则需要联网搜索
    if correct_count >= 3:
        return "analysis"
    else:
        return "web_search"


# 提供给外部使用的主节点字典和路由函数
NODES = {
    "start": start_node,
    "router": create_router_node,
    "simple_llm": simple_llm_node,
    "retrieval": create_retrieval_node,
    "evaluate": create_evaluate_node,
    "web_search": create_web_search_node,
    "analysis": create_analysis_node,
    "postprocess": create_postprocess_node,
    "memory_update": memory_update_node
}



In [ ]:
# instantiate_workflow.py
import json
from typing import Dict, List, Any

from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END


# 假设 LangGraph_nodes.py 中的定义已导入
# from LangGraph_nodes import (
#     AgentState,
#     start_node,
#     create_router_node,
#     simple_llm_node,          # 注意：此函数需要 state + llm 两个参数,需要包装
#     non_legal_node,
#     create_retrieval_node,
#     create_evaluate_node,
#     create_web_search_node,
#     create_analysis_node,
#     create_postprocess_node,
#     memory_update_node,
#     route_by_difficulty,
#     route_after_evaluation
# )


# ================== 1. 模拟依赖服务 ==================
class MockLLM:
    """模拟语言模型,简单返回固定内容"""

    def invoke(self, prompt, **kwargs):
        # 对于路由器,解析 prompt 中的 query 并返回假的 JSON
        # 这里极简处理：如果 query 中含有“天气”则判为非法,否则法律且简单
        if isinstance(prompt, str):
            text = prompt
        elif hasattr(prompt, 'text'):
            text = prompt.text
        else:
            text = str(prompt)
        if "天气" in text:
            return '{"is_legal": false, "difficulty": "不适用"}'
        else:
            return '{"is_legal": true, "difficulty": "简单"}'  # 默认可改为"困难"测试另一分支

    def __call__(self, *args, **kwargs):
        return self.invoke(*args, **kwargs)


class MockRAGService:
    """模拟检索服务,返回一些虚假文档"""

    def search_with_rerank(self, query, namespace, top_k=20, rerank_top_n=10):
        class FakeDoc:
            def __init__(self, id, text, score):
                self.id = id
                self.metadata = {"chunk_text": text}
                self.rerank_score = score

        return [
            FakeDoc(1, "合同法规定合同是双方或多方法律行为", 0.9),
            FakeDoc(2, "合同成立需要约与承诺", 0.8),
            FakeDoc(3, "无效合同的情形包括欺诈", 0.65),
            FakeDoc(4, "合同纠纷案例：A vs B", 0.3),
            FakeDoc(5, "与法律无关的文档片段", 0.1)
        ]


def mock_web_search(query: str, num: int) -> List[str]:
    return [f"网络搜索结果：关于“{query}”的补充法律资料片段{_}" for _ in range(num)]


def mock_pdf_generator(answer_text: str, filename: str) -> str:
    # 实际应用中会生成 PDF
    print(f"[模拟] PDF 已生成：{filename}")
    return f"/output/{filename}"


# ================== 2. 包装特殊节点函数 ==================
# simple_llm_node 需要 llm 参数,但 LangGraph 节点只能接收 state,故做闭包封装
def create_simple_llm_node(llm):
    def node(state: AgentState) -> dict:
        return simple_llm_node(state, llm)

    return node


# ================== 3. 构建状态图 ==================
def build_graph(llm, rag_service, search_func, pdf_generator):
    workflow = StateGraph(AgentState)

    # 实例化各节点（调用工厂或封装）
    router_node = create_router_node(llm)
    retrieval_node = create_retrieval_node(rag_service)
    evaluate_node = create_evaluate_node(llm)
    web_search_node = create_web_search_node(search_func)
    analysis_node = create_analysis_node(llm)
    postprocess_node = create_postprocess_node(pdf_generator)

    # 添加所有节点
    workflow.add_node("start", start_node)
    workflow.add_node("router", router_node)
    workflow.add_node("simple_llm", create_simple_llm_node(llm))
    workflow.add_node("non_legal", non_legal_node)
    workflow.add_node("retrieval", retrieval_node)
    workflow.add_node("evaluate", evaluate_node)
    workflow.add_node("web_search", web_search_node)
    workflow.add_node("analysis", analysis_node)
    workflow.add_node("postprocess", postprocess_node)
    # 可选：加入 memory_update 作为循环结束点
    # workflow.add_node("memory_update", memory_update_node)

    # 设置入口
    workflow.set_entry_point("start")

    # 添加边和条件边
    workflow.add_edge("start", "router")
    # 根据难度路由：非法 -> non_legal; 简单 -> simple_llm; 困难 -> retrieval
    workflow.add_conditional_edges(
        "router",
        route_by_difficulty,
        {
            "non_legal": "non_legal",
            "simple_llm": "simple_llm",
            "crag_retrieval": "retrieval"
        }
    )

    # retrieval 完成后执行评估
    workflow.add_edge("retrieval", "evaluate")
    # 评估后根据正确文档数量决定下一步
    workflow.add_conditional_edges(
        "evaluate",
        route_after_evaluation,
        {
            "analysis": "analysis",
            "web_search": "web_search"
        }
    )

    # 网络搜索后也进入分析
    workflow.add_edge("web_search", "analysis")

    # 分析后进入后处理,然后结束
    workflow.add_edge("analysis", "postprocess")
    workflow.add_edge("postprocess", END)

    # 简单法律问题和非法律问题直接结束
    workflow.add_edge("simple_llm", END)
    workflow.add_edge("non_legal", END)

    return workflow.compile()


# ================== 4. 运行示例 ==================
if __name__ == "__main__":
    # 初始化模拟依赖
    llm = MockLLM()
    rag = MockRAGService()
    search_fn = mock_web_search
    pdf_fn = mock_pdf_generator

    # 构建并编译图
    app = build_graph(llm, rag, search_fn, pdf_fn)

    # 准备初始状态
    initial_state = AgentState(
        query="什么是合同？",  # 可换成 "今天天气如何？" 测试非法问题
        messages=[],
        is_law_questions=False,
        is_simple_questions=False,
        rag_retrieved="",
        evaluation={"correct": [], "ambiguous": [], "incorrect": []},
        web_search_results=[],
        crag_context="",
        final_prompts="",
        final_answer="",
        is_pdf_output=False,
        should_loop=True
    )

    # 运行（同步模式）
    final_state = app.invoke(initial_state)

    print("\n=== 最终状态 ===")
    print("回答:", final_state.get("final_answer", final_state.get("answer", "无")))
    print("消息记录:")
    for msg in final_state.get("messages", []):
        print(f"  [{msg['role']}] {msg['content']}")